<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-09-24-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Daily Practice — 2026-09-24 — Statistics: Is Your Model's Error Real, or Just Noise Across Segments?

**Dataset:** [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) — 20,640 real 1990 U.S. Census block-group records (median income, house age, rooms, population, occupancy, lat/long), bundled with scikit-learn (`fetch_california_housing`). This is genuine census data, not a synthetic toy set.

## Problem statement

You're the statistics/QA reviewer on a team that just shipped a regression model predicting median
house value from census block-group features. The model's overall MAE looks acceptable, and everyone
is ready to sign off. But a stakeholder asks a harder question: **does the model make systematically
different errors depending on which income segment a block group falls into, and are the model's
underlying assumptions actually holding up?**

Eyeballing per-segment average error is not enough — a difference of $8,000 in mean residual between
two segments could be a real, actionable bias, or it could just be sampling noise from splitting the
data into groups. You need statistical tests that produce a p-value, not a vibe.

## What you should produce

A small, reusable statistical test suite (four functions) that a QA engineer could run against any
regression model release before shipping:

1. **`segment_bias_test(residuals, segment_labels)`** — tests whether the *mean* residual differs
   significantly across income-quartile segments, using **one-way ANOVA** (`scipy.stats.f_oneway`)
   as the primary test and **Kruskal-Wallis** (`scipy.stats.kruskal`) as a distribution-free
   cross-check, since residuals are rarely perfectly normal in practice.
2. **`residual_normality_test(residuals)`** — runs a **Shapiro-Wilk** test (`scipy.stats.shapiro`)
   on a random sample of residuals to check the "residuals are approximately normal" assumption that
   the ANOVA above leans on.
3. **`variance_homogeneity_test(residuals, segment_labels)`** — runs **Levene's test**
   (`scipy.stats.levene`) to check whether residual *variance* (not just the mean) is roughly equal
   across segments. Unequal variance means the model is far less reliable for some segments than
   others, even if the mean error looks fine.
4. **`error_concentration_test(residuals, segment_labels, threshold_quantile=0.90)`** — flags
   "large errors" as `|residual|` above a chosen percentile, builds a segment × large-error
   contingency table, and runs a **chi-square test of independence**
   (`scipy.stats.chi2_contingency`) to check whether large errors are disproportionately
   concentrated in one income segment rather than spread evenly.

Finally, wire the four functions together into a short **QA report** that prints each test's
statistic, p-value, a pass/fail decision at `alpha = 0.05`, and one plain-English sentence
interpreting what that means for shipping the model.


## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42
ALPHA = 0.05
rng = np.random.default_rng(RANDOM_STATE)


## Load the data

`MedInc` (median income in the block group, in tens of thousands of dollars) is used both as a
model feature and, binned into quartiles, as the "segment" we test for differential error. This
mirrors a common real-world QA question: *does this model work equally well for low-income and
high-income populations?*


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "median_house_value"})

FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "Latitude", "Longitude"]
TARGET = "median_house_value"

# Income quartile segment: Q1 = lowest-income block groups, Q4 = highest-income.
df["income_segment"] = pd.qcut(df["MedInc"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])

print(df.shape)
df[["MedInc", "income_segment", TARGET]].head()


In [ ]:
X_train, X_test, y_train, y_test, seg_train, seg_test = train_test_split(
    df[FEATURES], df[TARGET], df["income_segment"],
    test_size=0.25, random_state=RANDOM_STATE,
)

model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

pred_test = model.predict(X_test)
residuals = y_test.to_numpy() - pred_test

print(f"Test MAE: {mean_absolute_error(y_test, pred_test):.4f}")
print(f"Residual mean: {residuals.mean():.4f}, std: {residuals.std():.4f}")


## Your task

Fill in the four TODO functions below. Signatures are fixed — the "put it together" cell and the
solution assume these exact names and return shapes.


In [ ]:
def segment_bias_test(residuals, segment_labels):
    """
    TODO: Test whether mean residual differs significantly across segments.

    - Split `residuals` into one array per unique value of `segment_labels`.
    - Run `stats.f_oneway(*groups)` (one-way ANOVA) for the primary test.
    - Run `stats.kruskal(*groups)` (rank-based, doesn't assume normal residuals) as a cross-check.
    - Return a dict:
        {
            "anova_stat": ..., "anova_p": ...,
            "kruskal_stat": ..., "kruskal_p": ...,
        }
    """
    raise NotImplementedError


def residual_normality_test(residuals, sample_size=5000, rng=rng):
    """
    TODO: Check whether residuals are approximately normally distributed.

    - Shapiro-Wilk (`stats.shapiro`) is unreliable / very slow above a few thousand points, so
      if `len(residuals) > sample_size`, draw a random sample of that size without replacement
      (use `rng.choice`) before testing.
    - Return {"shapiro_stat": ..., "shapiro_p": ..., "n_tested": ...}.
    """
    raise NotImplementedError


def variance_homogeneity_test(residuals, segment_labels):
    """
    TODO: Test whether residual variance is equal across segments (homoscedasticity).

    - Split `residuals` into one array per unique value of `segment_labels`, same as above.
    - Run `stats.levene(*groups)` (Levene's test is robust to non-normal data, unlike Bartlett's).
    - Return {"levene_stat": ..., "levene_p": ...}.
    """
    raise NotImplementedError


def error_concentration_test(residuals, segment_labels, threshold_quantile=0.90):
    """
    TODO: Test whether "large errors" are concentrated in specific segments.

    - Compute `threshold = np.quantile(np.abs(residuals), threshold_quantile)`.
    - Build a boolean `is_large_error = np.abs(residuals) > threshold`.
    - Build a segment x {large_error, not_large_error} contingency table, e.g. with
      `pd.crosstab(segment_labels, is_large_error)`.
    - Run `stats.chi2_contingency(contingency_table)`.
    - Return {"chi2_stat": ..., "chi2_p": ..., "dof": ..., "threshold": threshold,
              "contingency_table": contingency_table}.
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. Call each of the four functions above on `residuals` / `seg_test.to_numpy()`.
# 2. For each test, decide pass ("no significant issue detected") vs. fail
#    ("statistically significant at alpha=0.05 -- investigate") by comparing the relevant
#    p-value(s) to ALPHA.
# 3. Print a short QA report: one line per test with the statistic, p-value, decision, and a
#    plain-English interpretation a reviewer could act on without knowing what a p-value is.


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def segment_bias_test(residuals, segment_labels):
    residuals = np.asarray(residuals)
    segment_labels = np.asarray(segment_labels)
    groups = [residuals[segment_labels == g] for g in np.unique(segment_labels)]

    anova_stat, anova_p = stats.f_oneway(*groups)
    kruskal_stat, kruskal_p = stats.kruskal(*groups)

    return {
        "anova_stat": anova_stat, "anova_p": anova_p,
        "kruskal_stat": kruskal_stat, "kruskal_p": kruskal_p,
    }


def residual_normality_test(residuals, sample_size=5000, rng=rng):
    residuals = np.asarray(residuals)
    if len(residuals) > sample_size:
        sample = rng.choice(residuals, size=sample_size, replace=False)
    else:
        sample = residuals

    shapiro_stat, shapiro_p = stats.shapiro(sample)
    return {"shapiro_stat": shapiro_stat, "shapiro_p": shapiro_p, "n_tested": len(sample)}


def variance_homogeneity_test(residuals, segment_labels):
    residuals = np.asarray(residuals)
    segment_labels = np.asarray(segment_labels)
    groups = [residuals[segment_labels == g] for g in np.unique(segment_labels)]

    levene_stat, levene_p = stats.levene(*groups)
    return {"levene_stat": levene_stat, "levene_p": levene_p}


def error_concentration_test(residuals, segment_labels, threshold_quantile=0.90):
    residuals = np.asarray(residuals)
    threshold = np.quantile(np.abs(residuals), threshold_quantile)
    is_large_error = np.abs(residuals) > threshold

    contingency_table = pd.crosstab(pd.Series(segment_labels, name="segment"), is_large_error)
    chi2_stat, chi2_p, dof, _expected = stats.chi2_contingency(contingency_table)

    return {
        "chi2_stat": chi2_stat, "chi2_p": chi2_p, "dof": dof,
        "threshold": threshold, "contingency_table": contingency_table,
    }
```

### Putting it together

```python
seg_test_arr = seg_test.to_numpy()

bias = segment_bias_test(residuals, seg_test_arr)
normality = residual_normality_test(residuals)
variance = variance_homogeneity_test(residuals, seg_test_arr)
concentration = error_concentration_test(residuals, seg_test_arr, threshold_quantile=0.90)

def verdict(p, alpha=ALPHA):
    return "FAIL (significant)" if p < alpha else "pass"

print("=== QA statistical test suite (alpha = 0.05) ===\n")

print(f"1. Segment bias (ANOVA):     stat={bias['anova_stat']:.3f}  p={bias['anova_p']:.4g}  -> {verdict(bias['anova_p'])}")
print(f"   Segment bias (Kruskal):   stat={bias['kruskal_stat']:.3f}  p={bias['kruskal_p']:.4g}  -> {verdict(bias['kruskal_p'])}")
print("   -> If FAIL: mean prediction error differs across income segments more than chance would\n"
      "      explain -- the model is systematically biased for at least one income group.\n")

print(f"2. Residual normality (Shapiro-Wilk): stat={normality['shapiro_stat']:.4f}  "
      f"p={normality['shapiro_p']:.4g} (n={normality['n_tested']})  -> {verdict(normality['shapiro_p'])}")
print("   -> If FAIL: residuals deviate from normal, so treat the ANOVA above as a rough signal\n"
      "      and trust the Kruskal-Wallis result more.\n")

print(f"3. Variance homogeneity (Levene):     stat={variance['levene_stat']:.3f}  "
      f"p={variance['levene_p']:.4g}  -> {verdict(variance['levene_p'])}")
print("   -> If FAIL: the model's error is far more variable (less reliable) in some income\n"
      "      segments than others, even if the average error looks similar.\n")

print(f"4. Large-error concentration (chi-square): stat={concentration['chi2_stat']:.3f}  "
      f"p={concentration['chi2_p']:.4g}  dof={concentration['dof']}  -> {verdict(concentration['chi2_p'])}")
print(f"   (large error = |residual| > {concentration['threshold']:.4f})")
print("   -> If FAIL: the model's worst mistakes are not spread evenly across income segments --\n"
      "      one segment absorbs a disproportionate share of the large errors.\n")
print(concentration["contingency_table"])
```

### Why this matters for ML QA

A single aggregate metric (overall MAE, overall accuracy) can hide the exact failure mode a
stakeholder cares about most: *does this model fail unevenly across a segment of the population?*
Reporting "Q4 has 12% higher mean residual than Q1" without a hypothesis test invites two opposite
mistakes — blocking a release over noise, or shipping a model with a real, actionable bias because
the gap "didn't look that big." Wrapping the comparison in ANOVA/Kruskal-Wallis (mean), Levene
(variance), and a chi-square test (tail-error concentration) turns "does this look off?" into "here
is the probability this pattern would occur under the null hypothesis of no segment effect,"
which is what a release-gating test suite should report.

</details>
